# Week 12 Demo: Neural Networks in PyTorch
This demo follows the Week 12 slides (*Intro to Artificial Neural Networks*). This notebook is where you **apply scikit-learn and PyTorch**: load real data, build models, write the training loop, evaluate, tune and save, the way you will in your own projects. The slides explain the ideas; every cell here uses PyTorch to do the work. Each section below is tied to a slide. When the slide deck says **"switch to the notebook"**, run that section, then go back to the slides.

| Section | Switch here (after slide) | Back to slides at | What we show |
|---|---|---|---|
| 1 | **13** Perceptron vs. Logistic Regression | 14 | scikit-learn `Perceptron` and `LogisticRegression`: fit, predict, weights, probabilities |
| 2 | **47** A Modern Classification MLP | 48 | scikit-learn `MLPRegressor` and `MLPClassifier` in a pipeline |
| 3 | **54** Inspecting the Model | 55 | Fashion MNIST, `nn.Sequential`, parameter counts, shapes, a Keras-to-PyTorch table |
| 4 | **60** Learning Curves | 61 | The **training loop** that replaces `fit()`, learning curves |
| 5 | **62** Making Predictions | 63 | Test evaluation and predictions |
| 6 | **64** Using the Regression Model | 65 | Regression MLP on California housing |
| 7 | **73** Dynamic by Default: Power vs. Cost | 74 | `nn.Module`, wide & deep, auxiliary output |
| 8 | **76** Checkpointing & Early Stopping | 77 | Saving, checkpointing, early stopping |
| 9 | **93** Learning Rate | 94 | The learning-rate range test |

Slide numbers count the title slide as slide 1.

<table align="left">
  <td>
    <a href="https://colab.research.google.com/github/pjmcswee/IST707-Notebooks/blob/main/week12/week12_pytorch_demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>
  </td>
</table>

## Setup
`torch` is PyTorch's tensor library (like NumPy, plus GPU support and **autograd**, which runs backpropagation for you). `torch.nn` holds the layers and losses. `torchvision` provides datasets such as Fashion MNIST. We use a GPU if there is one (on Colab: *Runtime, Change runtime type*), otherwise the CPU. Everything here runs fine on a CPU.

In [ ]:
import copy, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F

SEED = 42
torch.manual_seed(SEED); np.random.seed(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"
plt.rc("font", size=12)
print("torch", torch.__version__, "| device:", device)

: 

---
# ▶ Section 1: scikit-learn's `Perceptron` and `LogisticRegression`

**Switch here after slide 13 (Perceptron vs. Logistic Regression).** Back to the slides at 14.

Every scikit-learn model follows the same pattern: create it, `fit(X, y)`, then `predict(X_new)`. After fitting, the learned parameters are attributes ending in `_`. For a perceptron, `coef_` holds the TLU's weights **w** and `intercept_` its bias **b**.

In [ ]:
from sklearn.datasets import load_iris
from sklearn.linear_model import Perceptron

iris = load_iris(as_frame=True)
X = iris.data[["petal length (cm)", "petal width (cm)"]].values
y = (iris.target == 0).values            # Iris setosa vs. the rest

per_clf = Perceptron(random_state=42)
per_clf.fit(X, y)
print("predictions:", per_clf.predict([[2, 0.5], [3, 1]]))
print("weights w:", per_clf.coef_.round(2), " bias b:", per_clf.intercept_.round(2))

predictions: [ True False]
weights w: [[-1.4 -2.2]]  bias b: [4.]


Now a harder task: **Iris versicolor vs. the rest**, using all four features. These classes are **not linearly separable**, so no straight boundary separates them. Fit both linear models on a training split and score them on a test split. `make_pipeline(StandardScaler(), ...)` scales the features first, using the training data only.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

X = iris.data.values
y = (iris.target == 1).values            # Iris versicolor vs. the rest: NOT linearly separable
X_tr, X_te, y_tr, y_te = train_test_split(X, y, stratify=y, random_state=42)

per = make_pipeline(StandardScaler(), Perceptron(random_state=42)).fit(X_tr, y_tr)
log = make_pipeline(StandardScaler(), LogisticRegression()).fit(X_tr, y_tr)
print(f"Perceptron test accuracy:          {per.score(X_te, y_te):.3f}")
print(f"LogisticRegression test accuracy:  {log.score(X_te, y_te):.3f}")
print("LogisticRegression probabilities for 3 flowers:\n", log.predict_proba(X_te[:3]).round(2))
print("Perceptron has predict_proba?", hasattr(per, "predict_proba"))

Perceptron test accuracy:          0.684
LogisticRegression test accuracy:  0.684
LogisticRegression probabilities for 3 flowers:
 [[0.73 0.27]
 [0.82 0.18]
 [0.83 0.17]]
Perceptron has predict_proba? False


Both linear models top out at the same accuracy, because both can only draw a straight boundary. The difference is what they output: `LogisticRegression` has `predict_proba`, while the `Perceptron` only predicts a class.

**◀ Back to the slides: slide 14 (The XOR Problem).**

---
# ▶ Section 2: scikit-learn MLPs: `MLPRegressor` and `MLPClassifier`

**Switch here after slide 47 (A Modern Classification MLP).** Back to the slides at 48.

scikit-learn includes small multilayer perceptrons that use the same `fit` / `predict` API. `hidden_layer_sizes` lists the neurons in each hidden layer. MLPs are trained by gradient descent, so **always scale the inputs**: put a `StandardScaler` in the pipeline.

**Regression** on California housing: three hidden layers of 50 ReLU neurons, MSE loss, no output activation.

In [ ]:
from sklearn.datasets import fetch_california_housing
from sklearn.metrics import root_mean_squared_error
from sklearn.neural_network import MLPRegressor

housing = fetch_california_housing()
X_train_full, X_test, y_train_full, y_test = train_test_split(housing.data, housing.target, random_state=42)
X_train, X_valid, y_train, y_valid = train_test_split(X_train_full, y_train_full, random_state=42)

mlp_reg = MLPRegressor(hidden_layer_sizes=[50, 50, 50], random_state=42)
pipeline = make_pipeline(StandardScaler(), mlp_reg)
pipeline.fit(X_train, y_train)
rmse = root_mean_squared_error(y_valid, pipeline.predict(X_valid))
print(f"validation RMSE: {rmse:.3f}   (in $100,000s)")
print("training stopped after", mlp_reg.n_iter_, "epochs")

validation RMSE: 0.505   (in $100,000s)
training stopped after 169 epochs


**Classification** with `MLPClassifier`. First the versicolor-vs-rest task that defeated both linear models in Section 1, then all three species, where the output layer applies softmax.

In [ ]:
from sklearn.neural_network import MLPClassifier

# 1) The versicolor-vs-rest task that defeated both linear models above
X = iris.data.values
y = (iris.target == 1).values
X_tr, X_te, y_tr, y_te = train_test_split(X, y, stratify=y, random_state=42)
mlp = make_pipeline(StandardScaler(),
                    MLPClassifier(hidden_layer_sizes=[10], max_iter=2000, random_state=42))
mlp.fit(X_tr, y_tr)
print(f"versicolor vs. rest, MLP test accuracy: {mlp.score(X_te, y_te):.3f}   (linear models: 0.684)")

# 2) All three species: the output layer uses softmax
X_tr, X_te, y_tr, y_te = train_test_split(iris.data, iris.target, stratify=iris.target, random_state=42)
clf = make_pipeline(StandardScaler(),
                    MLPClassifier(hidden_layer_sizes=[10], max_iter=2000, random_state=42))
clf.fit(X_tr, y_tr)
print(f"3 species, MLP test accuracy: {clf.score(X_te, y_te):.3f}")
proba = clf.predict_proba(X_te[:3])
print("softmax probabilities for 3 flowers (each row sums to 1):\n", proba.round(3))

versicolor vs. rest, MLP test accuracy: 0.974   (linear models: 0.684)
3 species, MLP test accuracy: 0.947
softmax probabilities for 3 flowers (each row sums to 1):
 [[0.998 0.002 0.   ]
 [0.049 0.654 0.296]
 [0.018 0.942 0.039]]


The hidden layer lets the MLP draw a curved boundary, so it beats the linear models by a wide margin. `predict_proba` returns softmax probabilities that sum to 1 in every row.

scikit-learn's MLPs are handy for small tabular problems, but they offer only a fixed architecture, MSE or cross-entropy loss, and no GPU. For anything more, we switch to PyTorch.

**◀ Back to the slides: slide 48 (Typical Classification MLP).**

---
# ▶ Section 3: Fashion MNIST and the model, in PyTorch
**Switch here after slide 54 (Inspecting the Model).** Back to the slides at 55.

If you have seen **Keras** before (the book's TensorFlow edition, or online tutorials), this table maps its pieces to the PyTorch code used in the slides and this notebook.

| Keras | PyTorch (slides and this notebook) |
|---|---|
| `tf.keras.Sequential([...])` | `nn.Sequential(...)` |
| `Dense(300, activation="relu")` | `nn.Linear(784, 300)` then `nn.ReLU()` (you give the **input** size too) |
| `Flatten()` | `nn.Flatten()` |
| `Dense(10, activation="softmax")` | `nn.Linear(100, 10)` with **no softmax** (the loss applies it) |
| `model.summary()` | loop over `model.parameters()` (see below) |
| `compile(loss=..., optimizer=...)` | create `loss_fn` and `optimizer` objects |
| `sparse_categorical_crossentropy` | `nn.CrossEntropyLoss()` (takes **logits** and integer labels) |
| `model.fit(...)` | **you write the loop** (section 4) |
| `history.history` | a dictionary of lists you fill in |
| `model.evaluate` / `model.predict` | a loop under `model.eval()` and `torch.no_grad()` |
| `ModelCheckpoint`, `EarlyStopping` | a few lines in your loop (section 8) |
| `model.save("m.keras")` | `torch.save(model.state_dict(), "m.pt")` |

### Load and prepare the data (slide 52)
`torchvision` downloads Fashion MNIST for us. Like the slide, we hold out the last 5,000 training images for validation and scale pixels to 0 to 1.

In [ ]:
from torchvision import datasets
train_set = datasets.FashionMNIST("data", train=True, download=True)
test_set = datasets.FashionMNIST("data", train=False, download=True)

X_full, y_full = train_set.data.float() / 255., train_set.targets      # uint8 0-255 -> float 0-1
X_train, y_train = X_full[:-5000], y_full[:-5000]
X_valid, y_valid = X_full[-5000:], y_full[-5000:]
X_test, y_test = test_set.data.float() / 255., test_set.targets
X_train, y_train, X_valid, y_valid, X_test, y_test = [t.to(device) for t in (X_train, y_train, X_valid, y_valid, X_test, y_test)]

class_names = ["T-shirt/top", "Trouser", "Pullover", "Dress", "Coat",
               "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot"]
print("train", tuple(X_train.shape), "| valid", tuple(X_valid.shape), "| test", tuple(X_test.shape))
print("pixel range:", X_train.min().item(), "to", X_train.max().item(), "| labels are integers, e.g.", y_train[:5].tolist())

fig, axes = plt.subplots(2, 8, figsize=(14, 4))
for ax, img, lab in zip(axes.flat, X_train.cpu(), y_train.cpu()):
    ax.imshow(img, cmap="binary"); ax.set_title(class_names[lab], fontsize=9); ax.axis("off")
plt.tight_layout(); plt.show()

### Build the model (slide 53)
Same architecture as the slide: Flatten, then 300 ReLU units, then 100 ReLU units, then 10 outputs. Two things to notice: you state each layer's **input** size, and the last layer outputs raw scores (**logits**), not probabilities.

In [ ]:
torch.manual_seed(42)
model = nn.Sequential(
    nn.Flatten(),                 # [32, 28, 28] -> [32, 784], no parameters
    nn.Linear(784, 300), nn.ReLU(),
    nn.Linear(300, 100), nn.ReLU(),
    nn.Linear(100, 10),           # 10 logits, one per class
).to(device)
print(model)

### Inspect the model (slide 54)
There is no `summary()` built in. `named_parameters()` gives every parameter tensor with its name, so you can list the shapes and total the count yourself. Notice that each `nn.Linear` weight has shape **(out_features, in_features)**.

In [ ]:
total = 0
for name, p in model.named_parameters():
    print(f"{name:10s} shape {str(tuple(p.shape)):12s} {p.numel():>8,} parameters")
    total += p.numel()
print(f"{'total':10s} {'':12s} {total:>8,}")

In [ ]:
# Shape walk-through: one batch of 32 images
xb = X_train[:32]
print("input        :", tuple(xb.shape))
print("after Flatten:", tuple(model[0](xb).shape))
print("model output :", tuple(model(xb).shape), "  <- 32 instances x 10 class scores")

**◀ Back to the slides: slide 55 (Choosing the Loss and Optimizer).** Slides 55 to 59 choose a loss and optimizer and build the training loop. Section 4 runs it.

---
# ▶ Section 4: The training loop (the replacement for `fit()`)
**Switch here after slide 60 (Learning Curves).** Back to the slides at 61.

There is no `compile()` or `fit()`: we create a loss and an optimizer, then write the loop (slides 55 to 59).

**Loss (slides 55 and 56).** `nn.CrossEntropyLoss` (Keras: `sparse_categorical_crossentropy`) takes **raw logits** and **integer class labels**. It applies the softmax inside, which is also more numerically stable. That is why our model has no softmax layer. *If you add a softmax layer and still use `CrossEntropyLoss`, you apply softmax twice. The code runs and trains, just worse. You will see that bug in the exercise.*

**Optimizer.** `torch.optim.SGD(model.parameters(), lr=0.01)` is the optimizer on slide 55 (learning rate 0.01).

**Mini-batches.** A `DataLoader` hands out shuffled batches of 32 (slide 55: 55,000 / 32 = 1,719 batches per epoch).

In [ ]:
from torch.utils.data import DataLoader, TensorDataset

train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=32, shuffle=True)
valid_loader = DataLoader(TensorDataset(X_valid, y_valid), batch_size=256)
print("batches per epoch:", len(train_loader))

loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.01)

First, **one training step on one batch**, spelled out. These four calls are the core of every PyTorch training loop (slide 57):

In [ ]:
xb, yb = next(iter(train_loader))
logits = model(xb)                       # 1. forward pass
loss = loss_fn(logits, yb)               # 2. loss
optimizer.zero_grad()                    # clear old gradients: .grad accumulates across backward() calls
loss.backward()                          # 3. backward pass: gradients for all 266,610 parameters
optimizer.step()                         # 4. gradient descent step
print("loss on this batch:", round(loss.item(), 4), "  (ln 10 = 2.30 is what random guessing gives)")

Now wrap it. `run_epoch` (slide 58) makes one pass over a data loader. If you give it an optimizer it trains, otherwise it only evaluates. `fit` calls it once per epoch and prints the log shown on slide 59. The `loss` and `accuracy` for training are **running means** over the epoch, and `val_*` is measured once at the end.

In [ ]:
def run_epoch(model, loader, loss_fn, optimizer=None, classification=True):
    training = optimizer is not None
    model.train(training)                          # train mode vs eval mode (matters for dropout/batch norm in Week 13)
    total_loss, correct, n = 0.0, 0, 0
    with torch.set_grad_enabled(training):         # no gradient bookkeeping when evaluating
        for xb, yb in loader:
            out = model(xb)
            loss = loss_fn(out, yb)
            if training:
                optimizer.zero_grad(); loss.backward(); optimizer.step()
            total_loss += loss.item() * len(xb); n += len(xb)
            if classification:
                correct += (out.argmax(1) == yb).sum().item()
    return total_loss / n, (correct / n if classification else None)

def fit(model, loss_fn, optimizer, train_loader, valid_loader, epochs, classification=True, verbose=True):
    metric = "accuracy" if classification else "rmse"
    hist = {"loss": [], metric: [], "val_loss": [], "val_" + metric: []}
    for epoch in range(epochs):
        tr_loss, tr_acc = run_epoch(model, train_loader, loss_fn, optimizer, classification)
        va_loss, va_acc = run_epoch(model, valid_loader, loss_fn, None, classification)
        tr_m = tr_acc if classification else tr_loss ** 0.5       # regression metric: RMSE = sqrt(MSE)
        va_m = va_acc if classification else va_loss ** 0.5
        for k, v in zip(hist, (tr_loss, tr_m, va_loss, va_m)): hist[k].append(v)
        if verbose:
            print(f"Epoch {epoch+1:>2}/{epochs} - loss: {tr_loss:.4f} - {metric}: {tr_m:.4f}"
                  f" - val_loss: {va_loss:.4f} - val_{metric}: {va_m:.4f}")
    return hist

In [ ]:
t0 = time.time()
history = fit(model, loss_fn, optimizer, train_loader, valid_loader, epochs=30)
print(f"\ntraining took {time.time() - t0:.0f} s")

Your numbers may differ slightly from run to run, because random initialization and shuffling differ between machines (and GPU vs. CPU). `history` is a plain dict of lists, so pandas can plot the learning curves directly:

In [ ]:
pd.DataFrame(history).plot(figsize=(8, 5), xlim=[0, 29], ylim=[0, 1], grid=True, xlabel="Epoch")
plt.show()

Accuracy goes up and loss goes down, and the validation curves slowly separate from the training curves (mild overfitting). The validation loss is still falling, so the model hasn't converged. We could train longer.

**◀ Back to the slides: slide 61 (Tuning, Then Evaluating).**

---
# ▶ Section 5: Evaluate and predict
**Switch here after slide 62 (Making Predictions).** Back to the slides at 63.

**Test set, once** (slide 61: don't tune on the test set). `model.eval()` switches layers like dropout to evaluation behavior, and `torch.no_grad()` skips gradient bookkeeping.

In [ ]:
test_loader = DataLoader(TensorDataset(X_test, y_test), batch_size=256)
test_loss, test_acc = run_epoch(model, test_loader, loss_fn, None)
print(f"test loss {test_loss:.4f}  test accuracy {test_acc:.4f}")

**Predictions (slide 62).** The model outputs logits. To get probabilities apply `softmax`, and to get a class take the `argmax`. We predict the first three test images.

In [ ]:
model.eval()
with torch.no_grad():
    logits = model(X_test[:3])
    proba = F.softmax(logits, dim=1)
    y_pred = proba.argmax(dim=1)
print(proba.cpu().numpy().round(2))
print("predicted:", [class_names[i] for i in y_pred.tolist()])
print("actual:   ", [class_names[i] for i in y_test[:3].tolist()])

fig, axes = plt.subplots(1, 3, figsize=(7, 2.6))
for ax, img, p in zip(axes, X_test[:3].cpu(), y_pred.cpu()):
    ax.imshow(img, cmap="binary"); ax.set_title(class_names[p]); ax.axis("off")
plt.show()

**◀ Back to the slides: slide 63 (Regression MLP with PyTorch).**

---
# ▶ Section 6: A regression MLP
**Switch here after slide 64 (Using the Regression Model).** Back to the slides at 65.

An MLP for California housing: 3 hidden layers of 50 ReLU units, a **single output with no activation**, `nn.MSELoss`, the **Adam** optimizer, and RMSE as the metric. Standardizing the inputs matters a lot for gradient descent: compute the training mean and standard deviation and apply them to every split. We first clip outliers, explained in the code comment.

In [ ]:
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split

housing = fetch_california_housing()
Xh_full, Xh_test, yh_full, yh_test = train_test_split(housing.data, housing.target, random_state=42)
Xh_train, Xh_valid, yh_train, yh_valid = train_test_split(Xh_full, yh_full, random_state=42)

# Some columns have extreme outliers (average occupancy up to 230, population up to 16,305), which make plain
# standardization produce a few huge inputs and a jumpy training curve.  So we first clip each column to
# its 0.5 to 99.5 percentile range.  All statistics come from the TRAINING set only.
lo, hi = np.percentile(Xh_train, 0.5, axis=0), np.percentile(Xh_train, 99.5, axis=0)
clip = lambda a: np.clip(a, lo, hi)
mu, sigma = clip(Xh_train).mean(axis=0), clip(Xh_train).std(axis=0)
to_t = lambda a: torch.tensor(a, dtype=torch.float32, device=device)
Xh_train, Xh_valid, Xh_test = [to_t((clip(a) - mu) / sigma) for a in (Xh_train, Xh_valid, Xh_test)]
yh_train, yh_valid, yh_test = [to_t(a).view(-1, 1) for a in (yh_train, yh_valid, yh_test)]   # targets as a column
print("train", tuple(Xh_train.shape), "valid", tuple(Xh_valid.shape), "test", tuple(Xh_test.shape))

In [ ]:
torch.manual_seed(42)
reg = nn.Sequential(
    nn.Linear(8, 50), nn.ReLU(),
    nn.Linear(50, 50), nn.ReLU(),
    nn.Linear(50, 50), nn.ReLU(),
    nn.Linear(50, 1),                 # one output neuron, NO activation (slide 63)
).to(device)

reg_loader = DataLoader(TensorDataset(Xh_train, yh_train), batch_size=32, shuffle=True)
reg_valid = DataLoader(TensorDataset(Xh_valid, yh_valid), batch_size=512)
reg_hist = fit(reg, nn.MSELoss(), torch.optim.Adam(reg.parameters(), lr=1e-3), reg_loader, reg_valid,
               epochs=20, classification=False, verbose=False)
print("validation RMSE per epoch:", [round(v, 3) for v in reg_hist["val_rmse"][::4]], "...", round(reg_hist["val_rmse"][-1], 3))

The validation RMSE is in units of \$100,000, so 0.53 means roughly \$53,000 of typical error.

**A shape trap worth knowing:** the model's output has shape `(N, 1)`. If the targets have shape `(N,)`, `nn.MSELoss` *broadcasts* them into an `(N, N)` grid and silently computes the wrong thing (recent versions print a warning). That is why we reshaped the targets with `.view(-1, 1)` (slides 63 and 64).  Printing `.shape` on the model output and the targets before training catches it.

**◀ Back to the slides: slide 65 (Beyond Sequential).**

---
# ▶ Section 7: `nn.Module`, wide & deep, auxiliary output
**Switch here after slide 73 (Dynamic by Default: Power vs. Cost).** Back to the slides at 74.

**PyTorch has one idea: subclass `nn.Module`** (slide 67). You create layers in `__init__` and describe how data flows in `forward`. Because `forward` is ordinary Python, it is *dynamic* by default, with loops, `if` statements and `print` allowed (slide 73).

Here is the wide & deep network from slide 66, with **two inputs** and an **auxiliary output** (slide 71): features 0 to 4 go through the wide path and features 2 to 7 through the deep path, and the deep path feeds a regularizing side output.

In [ ]:
class WideAndDeep(nn.Module):
    def __init__(self, n_wide=5, n_deep=6, hidden=30):
        super().__init__()
        self.deep = nn.Sequential(nn.Linear(n_deep, hidden), nn.ReLU(),
                                  nn.Linear(hidden, hidden), nn.ReLU())
        self.main_out = nn.Linear(hidden + n_wide, 1)      # wide features are concatenated with the deep output
        self.aux_out = nn.Linear(hidden, 1)                # auxiliary output from the deep path only

    def forward(self, x_wide, x_deep):
        h = self.deep(x_deep)
        main = self.main_out(torch.cat([x_wide, h], dim=1))
        aux = self.aux_out(h)
        return main, aux

torch.manual_seed(42)
wd = WideAndDeep().to(device)
print(wd)
print("parameters:", sum(p.numel() for p in wd.parameters()))

In [ ]:
def split_inputs(X):                 # slide 68: wide = columns 0-4, deep = columns 2-7
    return X[:, :5], X[:, 2:]

opt = torch.optim.Adam(wd.parameters(), lr=1e-3)
mse = nn.MSELoss()
wd_loader = DataLoader(TensorDataset(Xh_train, yh_train), batch_size=32, shuffle=True)

for epoch in range(20):
    wd.train()
    for xb, yb in wd_loader:
        main, aux = wd(*split_inputs(xb))
        loss = 0.9 * mse(main, yb) + 0.1 * mse(aux, yb)    # slide 72: loss weights 0.9 main, 0.1 auxiliary
        opt.zero_grad(); loss.backward(); opt.step()
wd.eval()
with torch.no_grad():
    main, aux = wd(*split_inputs(Xh_valid))
print("validation RMSE  main output: %.3f   auxiliary output: %.3f"
      % (mse(main, yh_valid).sqrt().item(), mse(aux, yh_valid).sqrt().item()))

The main output beats the auxiliary one, as it should: the auxiliary head sees only the deep features. Because `forward` is plain Python, debugging is just `print` (try adding `print(h.shape)` inside `forward`), and dynamic behavior is easy. The cost is the one on slide 73: the model is less transparent to tools that expect a fixed graph. PyTorch accepts that trade-off by default.

**◀ Back to the slides: slide 74 (Saving, Checkpointing & TensorBoard).**

---
# ▶ Section 8: Saving, checkpointing, early stopping
**Switch here after slide 76 (Checkpointing & Early Stopping).** Back to the slides at 77.

**Saving (slide 75).** The PyTorch convention is to save the model's **`state_dict`**, a dictionary of all its parameter tensors. To restore, build a model with the **same architecture** and load the dictionary into it.

In [ ]:
torch.save(model.state_dict(), "fashion_mlp.pt")

restored = nn.Sequential(nn.Flatten(), nn.Linear(784, 300), nn.ReLU(),
                         nn.Linear(300, 100), nn.ReLU(), nn.Linear(100, 10)).to(device)
restored.load_state_dict(torch.load("fashion_mlp.pt", map_location=device))
restored.eval()
with torch.no_grad():
    same = torch.allclose(restored(X_test[:100]), model.eval()(X_test[:100]))
print("restored model gives identical outputs:", same)
print("state_dict keys:", list(model.state_dict().keys()))

**Checkpointing and early stopping (slide 76).** There are no callback objects. A few lines inside the loop do the job: remember the weights with the best validation loss, and stop when it hasn't improved for `patience` epochs, then restore the best weights. (In Keras these are the `ModelCheckpoint` and `EarlyStopping` callbacks.) We use Adam with a bigger learning rate so the model overfits quickly and we can see early stopping act.

In [ ]:
def fit_early_stopping(model, loss_fn, optimizer, train_loader, valid_loader, max_epochs=100, patience=5):
    best_loss, best_state, best_epoch, wait = float("inf"), None, 0, 0
    hist = {"loss": [], "val_loss": [], "val_accuracy": []}
    for epoch in range(max_epochs):
        tr_loss, _ = run_epoch(model, train_loader, loss_fn, optimizer)
        va_loss, va_acc = run_epoch(model, valid_loader, loss_fn)
        hist["loss"].append(tr_loss); hist["val_loss"].append(va_loss); hist["val_accuracy"].append(va_acc)
        if va_loss < best_loss:                                   # new best: "checkpoint" it in memory
            best_loss, best_epoch, wait = va_loss, epoch, 0
            best_state = copy.deepcopy(model.state_dict())
        else:
            wait += 1
            if wait >= patience:
                print(f"early stop at epoch {epoch + 1}; best was epoch {best_epoch + 1} (val_loss {best_loss:.4f})")
                break
    model.load_state_dict(best_state)                             # restore_best_weights=True
    return hist

torch.manual_seed(0)
es_model = nn.Sequential(nn.Flatten(), nn.Linear(784, 300), nn.ReLU(),
                         nn.Linear(300, 100), nn.ReLU(), nn.Linear(100, 10)).to(device)
es_hist = fit_early_stopping(es_model, nn.CrossEntropyLoss(), torch.optim.Adam(es_model.parameters(), lr=1e-3),
                             train_loader, valid_loader, patience=4)
pd.DataFrame(es_hist)[["loss", "val_loss"]].plot(figsize=(7, 4), grid=True, xlabel="epoch")
plt.show()
print("test accuracy of the restored best model: %.4f" % run_epoch(es_model, test_loader, nn.CrossEntropyLoss())[1])

Training loss keeps falling while validation loss bottoms out and turns up. That upturn is overfitting, and the stopped model is the one from the best epoch, not the last one.

**TensorBoard (slides 77 to 81).** PyTorch ships a writer for the same tool, `torch.utils.tensorboard.SummaryWriter`. Per epoch, call `writer.add_scalar("loss/val", va_loss, epoch)`. Then launch TensorBoard on the `runs/` directory as on slide 79. Slide 77 shows the extra lines in the loop; we skip running it here.

**◀ Back to the slides: slide 77 (TensorBoard).**

---
# ▶ Section 9: The learning-rate range test
**Switch here after slide 93 (Learning Rate).** Back to the slides at 94.

Slide 93: start with a tiny learning rate, multiply it by a constant factor every iteration up to about 10, and plot loss against learning rate on a log scale. The loss falls, then **shoots back up**. Pick a learning rate roughly 10 times below the turning point, re-initialize the model, and train normally.

In PyTorch we change the optimizer's learning rate through `optimizer.param_groups`.

In [ ]:
def lr_range_test(make_model, loader, lr_min=1e-5, lr_max=10.0, n_iter=500):
    model = make_model()
    opt = torch.optim.SGD(model.parameters(), lr=lr_min)
    factor = (lr_max / lr_min) ** (1 / n_iter)                  # slide 93's constant multiplier
    lrs, losses, best = [], [], float("inf")
    batches = iter(loader)
    for _ in range(n_iter):
        try: xb, yb = next(batches)
        except StopIteration: batches = iter(loader); xb, yb = next(batches)
        loss = F.cross_entropy(model(xb), yb)
        opt.zero_grad(); loss.backward(); opt.step()
        lrs.append(opt.param_groups[0]["lr"]); losses.append(loss.item())
        best = min(best, loss.item())
        if not np.isfinite(loss.item()) or loss.item() > 4 * best: break    # it blew up: stop early
        for g in opt.param_groups: g["lr"] *= factor
    return np.array(lrs), np.array(losses)

def fresh_model():
    torch.manual_seed(42)
    return nn.Sequential(nn.Flatten(), nn.Linear(784, 300), nn.ReLU(),
                         nn.Linear(300, 100), nn.ReLU(), nn.Linear(100, 10)).to(device)

lrs, losses = lr_range_test(fresh_model, train_loader)
smooth = pd.Series(losses).rolling(20, min_periods=1).mean()
turn = lrs[int(smooth.idxmin())]
print(f"loss is lowest near lr = {turn:.3g}; a good starting rate is about 10x below that: {turn / 10:.3g}")
plt.figure(figsize=(7, 4)); plt.semilogx(lrs, smooth); plt.axvline(turn / 10, color="g", ls="--", label="pick")
plt.axvline(turn, color="r", ls=":", label="turning point"); plt.xlabel("learning rate"); plt.ylabel("loss (smoothed)")
plt.legend(); plt.grid(alpha=.3); plt.show()

Slides 55 to 59 used `lr=0.01`. The test says the model could learn faster with a larger rate, which is why plain SGD at 0.01 looked slow (still improving after 30 epochs). The slide's warning applies: the best learning rate depends on the batch size and optimizer, so **re-tune it whenever you change one**.

**◀ Back to the slides: slide 94 (Batch Size, Activation, Iterations).**

---
# Cheat sheet: what to remember from this notebook
- **Data:** wrap tensors in a `TensorDataset` and a `DataLoader` (shuffle the training loader).
- **Training step** = `zero_grad()`, forward, loss, `backward()`, `step()`. Always in that spirit.
- **`nn.Linear(in, out)`** stores weights as `(out, in)`. **Logits in, `CrossEntropyLoss` out**: no softmax layer.
- **`model.train()` / `model.eval()`** and **`torch.no_grad()`** for evaluation.
- There is no built-in `fit`, callbacks or `summary`. You write the loop, so you control everything and must write the bookkeeping.
- Print `.shape` when something looks wrong, save the `state_dict`, and keep the test set for the end.